# Seizure/CSD Regression Workflow

This notebook analyzes seizure-like spike/peak activity before and after an induced CSD wave.

Main question:

> Which features predict total seizure duration before and after CSD?

Regression example:

```text
total_seizure_duration
=
spike_frequency
+
mean_spike_amplitude
+
burst_duration
+
burst_spike_count
+
condition
```

This tells us which seizure features are associated with longer seizure duration and whether pre/post CSD condition adds explanatory value.

## Required libraries

| Library | Purpose |
|---|---|
| `pathlib` | File/folder paths |
| `numpy` | Numeric calculations |
| `pandas` | Import and clean CSV tables |
| `matplotlib` | Plotting |
| `seaborn` | Statistical plots |
| `scikit-learn` | Train/test split, scaling, regression, metrics |
| `statsmodels` | Regression coefficients, p-values, confidence intervals |
| `joblib` | Save trained regression model |

Install:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn statsmodels joblib
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import statsmodels.formula.api as smf
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Input and output paths

Expected CSV:

```text
../data/seizure_csd_regression_data.csv
```

Expected columns:

| Column | Meaning |
|---|---|
| `animal_id` | Mouse identifier |
| `condition` | `pre_CSD` or `post_CSD` |
| `spike_frequency` | Spike/peak frequency |
| `mean_spike_amplitude` | Average spike/peak amplitude |
| `burst_duration` | Duration of high-frequency burst |
| `burst_spike_count` | Number of spikes/peaks in burst |
| `total_seizure_duration` | Outcome variable to predict |

In [ ]:
data_dir = Path("../data")
data_path = data_dir / "seizure_csd_regression_data.csv"

output_dir = Path("../outputs/seizure_csd_regression")
fig_dir = output_dir / "figures"
model_dir = output_dir / "models"

output_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

print("Input CSV:", data_path)
print("Output folder:", output_dir)

## Step 2 — Load data

Each row can represent one seizure-like event, burst, or analysis window.

In [ ]:
df = pd.read_csv(data_path)

print("Raw shape:", df.shape)
display(df.head())
print(df.columns.tolist())

## Step 3 — Clean column names

This cell accepts common alternate names such as:

- `frequency` → `spike_frequency`
- `amplitude` → `mean_spike_amplitude`
- `duration` → `total_seizure_duration`

In [ ]:
def standardize_columns(df):
    rename_map = {}

    for col in df.columns:
        clean = (
            col.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        if clean in ["mouse", "mouse_id", "animal", "animal_id"]:
            rename_map[col] = "animal_id"
        elif clean in ["group", "period", "pre_post", "condition"]:
            rename_map[col] = "condition"
        elif clean in ["frequency", "spike_freq", "peak_frequency", "spike_frequency"]:
            rename_map[col] = "spike_frequency"
        elif clean in ["amplitude", "mean_amplitude", "mean_spike_amplitude", "peak_amplitude"]:
            rename_map[col] = "mean_spike_amplitude"
        elif clean in ["burst_duration", "burst_length"]:
            rename_map[col] = "burst_duration"
        elif clean in ["burst_spikes", "burst_spike_count", "spike_count"]:
            rename_map[col] = "burst_spike_count"
        elif clean in ["duration", "total_duration", "seizure_duration", "total_seizure_duration"]:
            rename_map[col] = "total_seizure_duration"
        else:
            rename_map[col] = clean

    return df.rename(columns=rename_map)


df = standardize_columns(df)

required_cols = [
    "animal_id",
    "condition",
    "spike_frequency",
    "mean_spike_amplitude",
    "burst_duration",
    "burst_spike_count",
    "total_seizure_duration"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

numeric_cols = [
    "spike_frequency",
    "mean_spike_amplitude",
    "burst_duration",
    "burst_spike_count",
    "total_seizure_duration"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["animal_id"] = df["animal_id"].astype(str)
df["condition"] = df["condition"].astype(str)

df = df.dropna(subset=required_cols).copy()

df["condition"] = (
    df["condition"]
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Clean shape:", df.shape)
display(df.head())

## Step 4 — Exploratory summaries

This checks sample size, animal counts, and basic ranges before modeling.

In [ ]:
print("Condition counts:")
display(df["condition"].value_counts())

print("Animals per condition:")
display(
    df.groupby("condition")["animal_id"]
    .nunique()
    .rename("n_animals")
)

print("Descriptive statistics:")
display(df[numeric_cols].describe().round(4))

summary_path = output_dir / "regression_descriptive_summary.csv"
df[numeric_cols].describe().round(4).to_csv(summary_path)

print("Saved:", summary_path)

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="condition",
    y="total_seizure_duration"
)

sns.stripplot(
    data=df,
    x="condition",
    y="total_seizure_duration",
    color="black",
    alpha=0.35,
    size=3
)

plt.title("Total seizure duration before vs after CSD")
plt.tight_layout()

out_path = fig_dir / "total_seizure_duration_by_condition.tiff"

plt.savefig(
    out_path,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)

In [ ]:
predictor_cols = [
    "spike_frequency",
    "mean_spike_amplitude",
    "burst_duration",
    "burst_spike_count"
]

for predictor in predictor_cols:
    plt.figure(figsize=(6, 5))

    sns.scatterplot(
        data=df,
        x=predictor,
        y="total_seizure_duration",
        hue="condition"
    )

    sns.regplot(
        data=df,
        x=predictor,
        y="total_seizure_duration",
        scatter=False,
        color="black"
    )

    plt.title(f"{predictor} vs total seizure duration")
    plt.tight_layout()

    out_path = fig_dir / f"{predictor}_vs_total_seizure_duration.tiff"

    plt.savefig(
        out_path,
        dpi=300,
        format="tiff",
        bbox_inches="tight"
    )

    plt.show()

## Step 5 — Interpretable regression with statsmodels

This model estimates coefficients, p-values, confidence intervals, and R-squared.

Formula:

```text
total_seizure_duration ~ spike_frequency + mean_spike_amplitude + burst_duration + burst_spike_count + condition
```

What the output tells us:

| Output | Meaning |
|---|---|
| coefficient | Direction and size of association |
| p-value | Whether predictor is statistically associated with duration |
| confidence interval | Plausible range of coefficient |
| R-squared | Percent of variance explained |

In [ ]:
formula = (
    "total_seizure_duration ~ "
    "spike_frequency + "
    "mean_spike_amplitude + "
    "burst_duration + "
    "burst_spike_count + "
    "C(condition)"
)

ols_model = smf.ols(
    formula=formula,
    data=df
).fit()

print(ols_model.summary())

summary_text_path = output_dir / "ols_regression_summary.txt"

with open(summary_text_path, "w") as f:
    f.write(ols_model.summary().as_text())

print("Saved:", summary_text_path)

In [ ]:
coef_table = pd.DataFrame({
    "coefficient": ols_model.params,
    "p_value": ols_model.pvalues,
    "conf_low": ols_model.conf_int()[0],
    "conf_high": ols_model.conf_int()[1]
}).reset_index().rename(columns={"index": "term"})

coef_path = output_dir / "ols_regression_coefficients.csv"

coef_table.to_csv(coef_path, index=False)

display(coef_table)

print("Saved:", coef_path)

## Step 6 — Predictive regression with scikit-learn

This evaluates how well seizure duration can be predicted from spike/burst features.

Metrics:

| Metric | Meaning |
|---|---|
| MAE | Average absolute error |
| RMSE | Typical prediction error, penalizes large errors |
| R² | Variance explained on held-out test data |

In [ ]:
model_df = pd.get_dummies(
    df[
        [
            "condition",
            "spike_frequency",
            "mean_spike_amplitude",
            "burst_duration",
            "burst_spike_count",
            "total_seizure_duration"
        ]
    ],
    columns=["condition"],
    drop_first=True
)

X = model_df.drop(columns=["total_seizure_duration"])
y = model_df["total_seizure_duration"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

y_pred = linear_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame([{
    "model": "LinearRegression",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}])

display(metrics)

metrics.to_csv(
    output_dir / "regression_prediction_metrics.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(y_test, y_pred, alpha=0.75)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    color="black"
)

plt.xlabel("Observed total seizure duration")
plt.ylabel("Predicted total seizure duration")
plt.title("Observed vs Predicted Seizure Duration")
plt.tight_layout()

out_path = fig_dir / "observed_vs_predicted_duration.tiff"

plt.savefig(
    out_path,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()

print("Saved:", out_path)

## Step 7 — Ridge regression comparison

Ridge regression is useful if predictors are correlated, such as spike frequency, burst count, and burst duration.

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train, y_train)

ridge_pred = ridge_pipeline.predict(X_test)

ridge_metrics = pd.DataFrame([{
    "model": "RidgeRegression",
    "MAE": mean_absolute_error(y_test, ridge_pred),
    "RMSE": mean_squared_error(y_test, ridge_pred, squared=False),
    "R2": r2_score(y_test, ridge_pred)
}])

metrics_all = pd.concat(
    [metrics, ridge_metrics],
    ignore_index=True
)

display(metrics_all)

metrics_all.to_csv(
    output_dir / "regression_model_comparison_metrics.csv",
    index=False
)

## Step 8 — Save trained model

This saves the scikit-learn model so you can apply it to new seizure recordings later.

In [ ]:
model_path = model_dir / "seizure_duration_linear_regression_model.joblib"

joblib.dump(linear_pipeline, model_path)

feature_list_path = model_dir / "regression_model_input_features.txt"

with open(feature_list_path, "w") as f:
    for feature in X.columns:
        f.write(feature + "\n")

print("Saved model:", model_path)
print("Saved feature list:", feature_list_path)

## Overall output interpretation

This workflow provides two types of information.

### 1. Biological/statistical association

The statsmodels regression tells you which features are associated with longer seizure duration.

Examples:

- Higher spike frequency predicts longer seizure duration.
- Longer burst duration predicts longer total seizure duration.
- Post-CSD condition remains associated with duration even after accounting for spike frequency and burst features.

### 2. Predictive performance

The scikit-learn model tells you how accurately seizure duration can be predicted from spike/burst features.

Example:

```text
R² = 0.72
```

means the model explains 72% of variation in held-out seizure durations.

### Limitation

If there are many rows per animal, this basic regression may overestimate confidence because observations from the same animal are not fully independent.

Next-level models:

- mixed-effects regression
- Bayesian hierarchical regression
- time-series models